In [2]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from sklearn.impute import KNNImputer


In [7]:
pollutants = ["PM2.5 (µg/m³)", "PM10 (µg/m³)", "NO2 (µg/m³)", "SO2 (µg/m³)", "CO (mg/m³)", "Ozone (µg/m³)"]
max_gap_hours = 336  # Maximum gap length (in hours)
min_data_pct = 70    # Minimum percentage of data that must be present

sites_by_pollutant_gap = {}
sites_by_pollutant_missing = {}

for pollutant in pollutants:
    df = pd.read_csv(f"/home/rishi/ML Projects/Air Pollution/CPCB/visualize_2025/{pollutant.replace('/', '_').replace(' ', '_')}_df_cpcb_new_limits.csv",index_col=0, parse_dates=True)
    
    good_sites_gap = []
    good_sites_missing = []
    
    # Calculate missing percentage for each site
    missing_pct = df.isnull().sum(axis=0) * 100 / len(df)
    
    for site in df.columns:
        # Check 1: Missing data percentage
        if missing_pct[site] <= (100 - min_data_pct):
            good_sites_missing.append(site)
        
        # Check 2: Maximum gap length
        is_missing = df[site].isnull().values
        gap_lengths = []
        current_gap = 0
        for missing in is_missing:
            if missing:
                current_gap += 1
            else:
                if current_gap > 0:
                    gap_lengths.append(current_gap)
                    current_gap = 0
        if current_gap > 0:
            gap_lengths.append(current_gap)
        
        if len(gap_lengths) == 0 or np.max(gap_lengths) <= max_gap_hours:
            good_sites_gap.append(site)
    
    sites_by_pollutant_gap[pollutant] = set(good_sites_gap)
    sites_by_pollutant_missing[pollutant] = set(good_sites_missing)
    
    print(f"{pollutant}:")
    print(f"  - Max gap <= {max_gap_hours}h: {len(good_sites_gap)} sites")
    print(f"  - Missing <= {100-min_data_pct}%: {len(good_sites_missing)} sites")

# Find intersection across all pollutants for both conditions
sites_gap_filtered = set.intersection(*sites_by_pollutant_gap.values())
sites_missing_filtered = set.intersection(*sites_by_pollutant_missing.values())
sites = list(sites_gap_filtered.intersection(sites_missing_filtered))

print(f"\n{'='*60}")
print(f"Sites with max gap <= {max_gap_hours}h: {len(sites_gap_filtered)}")
print(f"Sites with missing <= {100-min_data_pct}%: {len(sites_missing_filtered)}")
print(f"Sites meeting BOTH criteria (all pollutants): {len(sites)}")
print(f"{'='*60}")

PM2.5 (µg/m³):
  - Max gap <= 336h: 387 sites
  - Missing <= 30%: 448 sites
PM10 (µg/m³):
  - Max gap <= 336h: 390 sites
  - Missing <= 30%: 442 sites
NO2 (µg/m³):
  - Max gap <= 336h: 395 sites
  - Missing <= 30%: 454 sites
SO2 (µg/m³):
  - Max gap <= 336h: 390 sites
  - Missing <= 30%: 443 sites
CO (mg/m³):
  - Max gap <= 336h: 397 sites
  - Missing <= 30%: 449 sites
Ozone (µg/m³):
  - Max gap <= 336h: 393 sites
  - Missing <= 30%: 427 sites

Sites with max gap <= 336h: 311
Sites with missing <= 30%: 369
Sites meeting BOTH criteria (all pollutants): 302


In [4]:
302*365*24*6

15873120

In [3]:
def clean(df, n_neighbors=5):
    df = df[['Timestamp', "PM2.5 (µg/m³)", "PM10 (µg/m³)", "NO2 (µg/m³)", "SO2 (µg/m³)", "CO (mg/m³)", "Ozone (µg/m³)"]]
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df = df.set_index('Timestamp')

    full_index = pd.date_range('2023-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')
    df = df.reindex(full_index)

    # Add temporal features to guide KNN distance calculations
    df['hour']       = df.index.hour
    df['dayofyear']  = df.index.dayofyear
    df['dayofweek']  = df.index.dayofweek

    feature_cols = ["PM2.5 (µg/m³)", "PM10 (µg/m³)", "NO2 (µg/m³)", "SO2 (µg/m³)", "CO (mg/m³)", "Ozone (µg/m³)"]

    imputer = KNNImputer(n_neighbors=n_neighbors)
    imputed = imputer.fit_transform(df)

    df_imputed = pd.DataFrame(imputed, index=df.index, columns=df.columns)
    df = df_imputed[feature_cols]

    return df

In [4]:
def process_site(site):
    """Process a single site: load, clean, and save"""
    path = r'/home/rishi/ML Projects/Air Pollution/CPCB/sites_comb'
    file = os.path.join(path, site)
    df = pd.read_csv(file)
    cleaned_df = clean(df)
    cleaned_df=cleaned_df.reset_index(names="Timestamp")
    os.makedirs('sites_imputed',exist_ok=True)
    cleaned_df.to_csv(fr'/home/rishi/ML Projects/Air Pollution/CPCB/sites_imputed/{site}', index=False)
    return site


with ProcessPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(process_site, site): site for site in sites}
    
    for future in tqdm(as_completed(futures), total=len(sites)):
        site= future.result()

100%|██████████| 138/138 [03:32<00:00,  1.54s/it]
